# SemEval-2026 Task 13 Subtask A: NB-SVM + Linear Blend v4

**Why this version?** The pseudo-label adaptation variant can overfit the 1,000 labeled `test_sample` examples because it uses them twice: first to define pseudo-label regions, then again to tune the final threshold.

This version switches to a **lower-variance sparse linear approach**:
1. Keep the original strong baseline: **SGD logistic** on structural + TF-IDF features
2. Add **NB-SVM style log-count-ratio reweighting** on the text TF-IDF block
3. Compare both models and a simple probability blend
4. Use `test_sample` only for **model/threshold selection and prior correction**, not for pseudo-labeling

NB-SVM is often stronger on sparse text/code classification because it upweights class-indicative ngrams before the linear classifier sees them.

In [1]:
import pandas as pd
import numpy as np
import re
import gc
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, classification_report
from sklearn.calibration import CalibratedClassifierCV
from scipy.sparse import hstack, csr_matrix
from joblib import Parallel, delayed

print('All imports OK')

All imports OK


## 1. Load Data

In [2]:
import os, glob

# Auto-discover the correct data path
_candidates = [
    '/kaggle/input/semeval-2026-task-13-subtask-a',
    '/kaggle/input/semeval-2026-task-13-subtask-a/Task_A',
    '/kaggle/input/sem-eval-2026-task-13-subtask-a/Task_A',
]
DATA_DIR = None
for _c in _candidates:
    if os.path.exists(os.path.join(_c, 'train.parquet')):
        DATA_DIR = _c
        break
if DATA_DIR is None:
    # Last resort: search
    _found = glob.glob('/kaggle/input/**/train.parquet', recursive=True)
    if _found:
        DATA_DIR = os.path.dirname(_found[0])
print(f'DATA_DIR: {DATA_DIR}')
print('Files:', os.listdir(DATA_DIR))

train       = pd.read_parquet(f'{DATA_DIR}/train.parquet')
val         = pd.read_parquet(f'{DATA_DIR}/validation.parquet')
test        = pd.read_parquet(f'{DATA_DIR}/test.parquet')
test_sample = pd.read_parquet(f'{DATA_DIR}/test_sample.parquet')

for df in [train, val, test, test_sample]:
    df['code'] = df['code'].fillna('')

print(f'Train: {train.shape}, Val: {val.shape}, Test: {test.shape}, test_sample: {test_sample.shape}')
print(f'Train label dist: {train["label"].value_counts().to_dict()}')
print(f'Val label dist:   {val["label"].value_counts().to_dict()}')
print(f'test_sample dist: {test_sample["label"].value_counts().to_dict()}')

# Prior probabilities
train_prior_ai = train['label'].mean()          # ~0.52
test_prior_ai  = test_sample['label'].mean()    # ~0.22
print(f'\nTrain prior P(AI): {train_prior_ai:.3f}')
print(f'Test prior  P(AI): {test_prior_ai:.3f}  <-- significant shift!')

DATA_DIR: /kaggle/input/competitions/sem-eval-2026-task-13-subtask-a/Task_A
Files: ['sample_submission.csv', 'train.parquet', 'test_sample.parquet', 'validation.parquet', 'test.parquet']
Train: (500000, 4), Val: (100000, 4), Test: (500000, 2), test_sample: (1000, 4)
Train label dist: {1: 261525, 0: 238475}
Val label dist:   {1: 52305, 0: 47695}
test_sample dist: {0: 777, 1: 223}

Train prior P(AI): 0.523
Test prior  P(AI): 0.223  <-- significant shift!


## 2. Feature Engineering

Extended feature set based on EDA insights:
- Indentation features are the strongest structural predictors (std_indent top-1)
- TF-IDF dominates (84% of importance) but structural features matter for calibration
- Added: operator density, string literal counts, type annotation patterns

In [3]:
def _process_single(text):
    if not isinstance(text, str) or len(text) == 0:
        return [0.0] * 28

    lines = text.split('\n')
    n_lines = len(lines)
    non_empty = [l for l in lines if l.strip()]

    # --- Length features ---
    code_len     = len(text)
    log_len      = np.log1p(code_len)
    avg_line_len = np.mean([len(l) for l in lines]) if lines else 0.0
    max_line_len = max((len(l) for l in lines), default=0)
    n_tokens     = len(re.findall(r'\w+', text))

    # --- Blank / comment lines ---
    n_blank       = sum(1 for l in lines if l.strip() == '')
    n_comment     = sum(1 for l in lines if l.strip().startswith(('#', '//')))
    blank_ratio   = n_blank   / max(n_lines, 1)
    comment_ratio = n_comment / max(n_lines, 1)

    # --- Indentation (strongest signal: std_indent, max_indent, avg_indent) ---
    if non_empty:
        indents      = [len(l) - len(l.lstrip()) for l in non_empty]
        avg_indent   = float(np.mean(indents))
        max_indent   = float(max(indents))
        std_indent   = float(np.std(indents))
        median_indent = float(np.median(indents))
    else:
        avg_indent = max_indent = std_indent = median_indent = 0.0

    # --- Docstrings (AI 18% vs Human 0.6%) ---
    docstring_matches = re.findall(r'"""[\s\S]*?"""|\'{3}[\s\S]*?\'{3}', text)
    has_docstring  = int(len(docstring_matches) > 0)
    n_docstrings   = len(docstring_matches)
    docstring_len  = sum(len(d) for d in docstring_matches)

    # --- Code structure ---
    n_functions = len(re.findall(r'\bdef\b|\bfunction\b', text))
    n_classes   = len(re.findall(r'\bclass\b', text))
    n_imports   = len(re.findall(r'^\s*(?:import|from)\s', text, re.MULTILINE))
    n_returns   = len(re.findall(r'\breturn\b', text))

    # --- Type annotations (Python: AI tends to add them) ---
    n_type_hints = len(re.findall(r'->|:\s*(?:int|str|float|bool|list|dict|tuple|set|Any|Optional|Union|List|Dict)', text))

    # --- Vocabulary diversity ---
    all_tokens   = re.findall(r'\w+', text)
    unique_ratio = len(set(all_tokens)) / max(len(all_tokens), 1)
    semicolons   = text.count(';')

    # --- Natural language word ratio (AI: more English prose in comments/docstrings) ---
    english_words = len(re.findall(
        r'\b(?:the|is|are|this|that|for|with|returns?|param|function|'
        r'method|class|object|value|result|output|input|data|list|string|'
        r'number|type|if|else|while|loop|index|array|dict|key|true|false|'
        r'none|null|void|bool|int|float|str|char|size|length|count|name|'
        r'error|exception|check|valid|create|update|delete|get|set|add|'
        r'remove|find|sort|compare|convert|print|read|write|open|close|'
        r'file|path|directory|module|package|library|test|main|init|run)\b',
        text.lower()
    ))
    english_ratio = english_words / max(n_tokens, 1)

    # --- Operator density ---
    operators = len(re.findall(r'[+\-*/=<>!&|^~%]{1,3}', text))
    op_density = operators / max(n_tokens, 1)

    # --- Trailing whitespace (style signal) ---
    trailing_ws = sum(1 for l in lines if l != l.rstrip()) / max(n_lines, 1)

    return [
        code_len, log_len, avg_line_len, max_line_len, n_lines, n_tokens,
        blank_ratio, comment_ratio,
        avg_indent, max_indent, std_indent, median_indent,
        has_docstring, n_docstrings, docstring_len,
        n_functions, n_classes, n_imports, n_returns, n_type_hints,
        unique_ratio, semicolons, english_words, english_ratio,
        operators, op_density, trailing_ws,
        code_len / max(n_lines, 1),   # avg chars per line (different from avg_line_len due to blank lines)
    ]


FEAT_NAMES = [
    'code_len', 'log_len', 'avg_line_len', 'max_line_len', 'n_lines', 'n_tokens',
    'blank_ratio', 'comment_ratio',
    'avg_indent', 'max_indent', 'std_indent', 'median_indent',
    'has_docstring', 'n_docstrings', 'docstring_len',
    'n_functions', 'n_classes', 'n_imports', 'n_returns', 'n_type_hints',
    'unique_ratio', 'semicolons', 'english_words', 'english_ratio',
    'operators', 'op_density', 'trailing_ws',
    'chars_per_line',
]


def extract_features(df, n_jobs=4):
    rows = Parallel(n_jobs=n_jobs, backend='loky')(
        delayed(_process_single)(text) for text in df['code']
    )
    return pd.DataFrame(rows, columns=FEAT_NAMES)


print(f'Feature function defined ({len(FEAT_NAMES)} features)')
print('Extracting train features...')
train_feats = extract_features(train)
print(f'Train features shape: {train_feats.shape}')

Feature function defined (28 features)
Extracting train features...
Train features shape: (500000, 28)


In [4]:
print('Extracting val, test, test_sample features...')
val_feats = extract_features(val)
test_feats = extract_features(test)
ts_feats = extract_features(test_sample)
print('Done')

Extracting val, test, test_sample features...
Done


## 3. TF-IDF on Code Tokens

In [5]:
def code_tokenizer(text):
    return re.findall(r'[a-zA-Z_][a-zA-Z0-9_]*|\d+|[+\-*/=<>!&|^~%]+|[{}()\[\];,.:]+', text)


print('Fitting word TF-IDF...')
tfidf_word = TfidfVectorizer(
    tokenizer=code_tokenizer,
    max_features=20000,
    min_df=5,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32
)
train_tfidf_w = tfidf_word.fit_transform(train['code'])
val_tfidf_w   = tfidf_word.transform(val['code'])
test_tfidf_w  = tfidf_word.transform(test['code'])
ts_tfidf_w    = tfidf_word.transform(test_sample['code'])
print(f'Word TF-IDF shape: {train_tfidf_w.shape}')

print('Fitting char TF-IDF...')
tfidf_char = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 4),
    max_features=12000,
    min_df=5,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32
)
train_tfidf_c = tfidf_char.fit_transform(train['code'])
val_tfidf_c   = tfidf_char.transform(val['code'])
test_tfidf_c  = tfidf_char.transform(test['code'])
ts_tfidf_c    = tfidf_char.transform(test_sample['code'])
print(f'Char TF-IDF shape: {train_tfidf_c.shape}')

Fitting word TF-IDF...
Word TF-IDF shape: (500000, 20000)
Fitting char TF-IDF...
Char TF-IDF shape: (500000, 12000)


## 4. Build Sparse Blocks

For NB-SVM we keep **structural features separate** from the TF-IDF text block.
- Base model uses: `[structural | raw TF-IDF]`
- NB-SVM model uses: `[structural | TF-IDF * log-count-ratio]`

In [ ]:

from scipy.sparse import csr_matrix, hstack

def to_struct_block(feats_df):
    return csr_matrix(feats_df.values.astype(np.float32))

X_train_struct = to_struct_block(train_feats)
X_val_struct   = to_struct_block(val_feats)
X_test_struct  = to_struct_block(test_feats)
X_ts_struct    = to_struct_block(ts_feats)

X_train_text = hstack([train_tfidf_w, train_tfidf_c], format='csr')
X_val_text   = hstack([val_tfidf_w,   val_tfidf_c],   format='csr')
X_test_text  = hstack([test_tfidf_w,  test_tfidf_c],  format='csr')
X_ts_text    = hstack([ts_tfidf_w,    ts_tfidf_c],    format='csr')

y_train = train['label'].values.astype(np.int32)
y_val   = val['label'].values.astype(np.int32)
y_ts    = test_sample['label'].values.astype(np.int32)

print(f'Structural features: {X_train_struct.shape[1]}')
print(f'Text features:       {X_train_text.shape[1]}')
print(f'Total features:      {X_train_struct.shape[1] + X_train_text.shape[1]}')

# v4 previously kept raw TF-IDF blocks, combined text blocks, and both base/NB matrices
# alive at the same time. That can kill the Kaggle kernel on the full test split.
# Keep only the minimal sparse blocks that we will reuse.
del train_tfidf_w, train_tfidf_c, val_tfidf_w, val_tfidf_c
del test_tfidf_w, test_tfidf_c, ts_tfidf_w, ts_tfidf_c
del train_feats, val_feats, test_feats, ts_feats
gc.collect()


## 5. Train the Original Sparse Logistic Baseline

This is the real baseline in the prior notebook (despite the Random Forest title).

In [ ]:

import time
from sklearn.linear_model import SGDClassifier

base_clf = SGDClassifier(
    loss='log_loss',
    alpha=1e-5,
    class_weight='balanced',
    max_iter=3000,
    random_state=42
)

print('Training base sparse logistic model...')
t0 = time.time()
X_train_base = hstack([X_train_struct, X_train_text], format='csr')
base_clf.fit(X_train_base, y_train)
print(f'Base training done in {(time.time()-t0)/60:.1f} min')

# Validation / test_sample probabilities are small enough to score directly.
X_val_base = hstack([X_val_struct, X_val_text], format='csr')
X_ts_base  = hstack([X_ts_struct,  X_ts_text],  format='csr')
base_val_probs = base_clf.predict_proba(X_val_base)[:, 1].astype(np.float32)
base_ts_probs  = base_clf.predict_proba(X_ts_base)[:, 1].astype(np.float32)

del X_train_base, X_val_base, X_ts_base
gc.collect()


## 6. Build NB-SVM Features

Traditional NB-SVM computes a per-feature **log-count-ratio** between the positive and negative classes, then multiplies the sparse text matrix by that ratio before fitting a linear classifier.

Here we apply that transform only to the **text block**. Structural features are appended unchanged.

In [ ]:

def compute_log_count_ratio(X, y, alpha=1.0):
    """
    Binary NB log-count-ratio for sparse non-negative features.
    r_j = log( P(feature_j | AI) / P(feature_j | Human) )
    """
    pos = X[y == 1]
    neg = X[y == 0]

    p = np.asarray(pos.sum(axis=0)).ravel().astype(np.float64) + alpha
    q = np.asarray(neg.sum(axis=0)).ravel().astype(np.float64) + alpha

    p /= p.sum()
    q /= q.sum()
    r = np.log(p) - np.log(q)
    return r.astype(np.float32)

r_text = compute_log_count_ratio(X_train_text, y_train, alpha=1.0)
print('Computed NB log-count-ratio vector:', r_text.shape)
print('ratio stats:', float(r_text.min()), float(r_text.mean()), float(r_text.max()))

# Only materialize the transformed matrices when needed.
X_train_nb = hstack([X_train_struct, X_train_text.multiply(r_text)], format='csr')
X_val_nb   = hstack([X_val_struct,   X_val_text.multiply(r_text)],   format='csr')
X_ts_nb    = hstack([X_ts_struct,    X_ts_text.multiply(r_text)],    format='csr')

print(f'NB-SVM train matrix: {X_train_nb.shape}')


In [ ]:

nbsvm_clf = SGDClassifier(
    loss='log_loss',
    alpha=5e-6,
    class_weight='balanced',
    max_iter=3000,
    random_state=42
)

print('Training NB-SVM-style linear model...')
t0 = time.time()
nbsvm_clf.fit(X_train_nb, y_train)
print(f'NB-SVM training done in {(time.time()-t0)/60:.1f} min')

nb_val_probs = nbsvm_clf.predict_proba(X_val_nb)[:, 1].astype(np.float32)
nb_ts_probs  = nbsvm_clf.predict_proba(X_ts_nb)[:, 1].astype(np.float32)

del X_train_nb, X_val_nb, X_ts_nb
gc.collect()


## 7. Compare Base, NB-SVM, and a Simple Probability Blend

We evaluate three candidates on `test_sample`:
- `base` = raw TF-IDF sparse logistic
- `nbsvm` = NB-reweighted sparse logistic
- `blend_w` = `(1-w) * base + w * nbsvm` for coarse weights

For each candidate we test both:
- direct threshold tuning
- prior-corrected threshold tuning

Then we keep the single best candidate.

In [ ]:

def prior_correct(probs, train_prior_ai, test_prior_ai):
    odds = probs / (1.0 - probs + 1e-10)
    ratio = (test_prior_ai / (1.0 - test_prior_ai)) / (train_prior_ai / (1.0 - train_prior_ai))
    odds_corr = odds * ratio
    return odds_corr / (1.0 + odds_corr)

def tune_threshold(y_true, probs, grid):
    best_f1, best_t = -1.0, 0.5
    for t in grid:
        pred = (probs > t).astype(np.int32)
        f1 = f1_score(y_true, pred, average='macro')
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_f1, best_t

candidates = {
    'base': (base_val_probs, base_ts_probs),
    'nbsvm': (nb_val_probs, nb_ts_probs),
}

for w in np.arange(0.1, 1.0, 0.1):
    name = f'blend_{w:.1f}'
    val_probs = ((1.0 - w) * base_val_probs + w * nb_val_probs).astype(np.float32)
    ts_probs = ((1.0 - w) * base_ts_probs + w * nb_ts_probs).astype(np.float32)
    candidates[name] = (val_probs, ts_probs)

rows = []
for name, (val_probs, ts_probs) in candidates.items():
    direct_f1, direct_t = tune_threshold(y_ts, ts_probs, np.arange(0.10, 0.90, 0.01))
    ts_probs_corr = prior_correct(ts_probs, train_prior_ai, test_prior_ai)
    corr_f1, corr_t = tune_threshold(y_ts, ts_probs_corr, np.arange(0.05, 0.95, 0.005))
    strategy = 'prior_corrected' if corr_f1 >= direct_f1 else 'direct_threshold'
    best_f1 = corr_f1 if corr_f1 >= direct_f1 else direct_f1
    best_t  = corr_t if corr_f1 >= direct_f1 else direct_t
    pred_for_report = (ts_probs_corr > corr_t).astype(np.int32) if strategy == 'prior_corrected' else (ts_probs > direct_t).astype(np.int32)
    ai_rate = pred_for_report.mean()
    rows.append({
        'candidate': name,
        'direct_f1': direct_f1,
        'direct_thresh': direct_t,
        'corr_f1': corr_f1,
        'corr_thresh': corr_t,
        'chosen_strategy': strategy,
        'chosen_f1': best_f1,
        'chosen_thresh': best_t,
        'pred_ai_rate': ai_rate,
    })

leaderboard = pd.DataFrame(rows).sort_values('chosen_f1', ascending=False).reset_index(drop=True)
print(leaderboard.head(12).to_string(index=False))

best = leaderboard.iloc[0].to_dict()
best_name = best['candidate']
best_strategy = best['chosen_strategy']
best_thresh = float(best['chosen_thresh'])
best_f1 = float(best['chosen_f1'])

print('\nSelected candidate:')
print(best)


In [ ]:
best_val_probs, best_ts_probs = candidates[best_name]
if best_strategy == 'prior_corrected':
    best_ts_probs_eval = prior_correct(best_ts_probs, train_prior_ai, test_prior_ai)
else:
    best_ts_probs_eval = best_ts_probs

best_ts_preds = (best_ts_probs_eval > best_thresh).astype(np.int32)
print(f'Best candidate on test_sample: {best_name}')
print(f'Strategy: {best_strategy}')
print(f'test_sample Macro F1: {best_f1:.4f}')
print(classification_report(y_ts, best_ts_preds, target_names=['Human', 'AI']))

## 8. Generate Submission

We score the full test set with the selected candidate, apply the selected probability strategy, then write `submission.csv`.

In [ ]:

def predict_probs_in_batches(clf, X_struct, X_text, batch_size=50000, r_text=None):
    probs = np.empty(X_struct.shape[0], dtype=np.float32)
    n = X_struct.shape[0]
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        Xb_text = X_text[start:end]
        if r_text is not None:
            Xb_text = Xb_text.multiply(r_text)
        Xb = hstack([X_struct[start:end], Xb_text], format='csr')
        probs[start:end] = clf.predict_proba(Xb)[:, 1].astype(np.float32)
        del Xb_text, Xb
        if start % (batch_size * 4) == 0:
            gc.collect()
    gc.collect()
    return probs

print('Scoring full test in batches to avoid kernel OOM...')
base_test_probs = predict_probs_in_batches(base_clf,  X_test_struct, X_test_text, batch_size=50000)
nb_test_probs   = predict_probs_in_batches(nbsvm_clf, X_test_struct, X_test_text, batch_size=50000, r_text=r_text)

if best_name == 'base':
    test_probs = base_test_probs
elif best_name == 'nbsvm':
    test_probs = nb_test_probs
else:
    w = float(best_name.split('_')[1])
    test_probs = ((1.0 - w) * base_test_probs + w * nb_test_probs).astype(np.float32)

if best_strategy == 'prior_corrected':
    test_probs_final = prior_correct(test_probs, train_prior_ai, test_prior_ai)
else:
    test_probs_final = test_probs

test_preds = (test_probs_final > best_thresh).astype(np.int32)

print(f'Test predictions (candidate={best_name}, strategy={best_strategy}, thresh={best_thresh:.3f}):')
print(f'  Human (0): {(test_preds == 0).sum():,} ({(test_preds == 0).mean()*100:.1f}%)')
print(f'  AI (1):    {(test_preds == 1).sum():,} ({(test_preds == 1).mean()*100:.1f}%)')
print(f'  Expected AI%: ~22% based on test_sample')

# Free large blocks before submission / notebook conversion
del X_train_struct, X_val_struct, X_ts_struct
del X_train_text, X_val_text, X_ts_text
del base_test_probs, nb_test_probs
gc.collect()


In [ ]:
test_ids = pd.read_parquet(f'{DATA_DIR}/test.parquet', columns=['ID'])

submission = pd.DataFrame({
    'ID': test_ids['ID'],
    'label': test_preds
})
submission.to_csv('submission.csv', index=False)

print(f'Submission saved: {submission.shape}')
print(submission['label'].value_counts())
print(submission.head())

In [ ]:
!kaggle competitions submit -c sem-eval-2026-task-13-subtask-a -f submission.csv -m "NB-SVM + sparse blend v4"